In [1]:
import sqlite3
import warnings
import os
from pathlib import Path

import pandas as pd
warnings.filterwarnings("ignore")

In [2]:
# Création des tables et injection des données

db_path = "../irve_database.db"
csv_path = "../data/processed/dataset_irve_clean.csv"

# Suppression de base existante
if os.path.exists(db_path):
    try:
        os.remove(db_path)
        print("Ancienne base de données supprimée.")
    except Exception as e:
        print("Note : Reconnectez le kernel si le fichier est verrouillé.")

# Chargement du dataframe
df_clean = pd.read_csv(csv_path, low_memory=False)

# Connexion à SQLite
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

# Création des 4 tables (Schéma DDL)
cursor.executescript("""
CREATE TABLE COMMUNES (
    code_insee TEXT PRIMARY KEY,
    nom_commune TEXT NOT NULL,
    code_postal TEXT,
    code_departement TEXT
);

CREATE TABLE OPERATEURS (
    id_operateur INTEGER PRIMARY KEY AUTOINCREMENT,
    nom_amenageur TEXT,
    nom_operateur TEXT
);

CREATE TABLE STATIONS (
    id_station TEXT PRIMARY KEY,
    code_insee TEXT NOT NULL,
    id_operateur INTEGER NOT NULL,
    nom_station TEXT,
    adresse_station TEXT,
    latitude REAL,
    longitude REAL,
    nbre_pdc INTEGER,
    FOREIGN KEY (code_insee) REFERENCES COMMUNES(code_insee),
    FOREIGN KEY (id_operateur) REFERENCES OPERATEURS(id_operateur)
);

CREATE TABLE POINTS_DE_CHARGE (
    id_pdc TEXT PRIMARY KEY,
    id_station TEXT NOT NULL,
    puissance_nominale REAL,
    tranche_puissance TEXT,
    prise_type_2 BOOLEAN,
    prise_type_combo_ccs BOOLEAN,
    FOREIGN KEY (id_station) REFERENCES STATIONS(id_station)
);
""")

print("Schéma de la base de données créé avec succès.")

# INJECTION TABLE 1 : COMMUNES (Parente)

df_communes = df_clean[
    ["code_insee_commune", "consolidated_commune", "consolidated_code_postal", "code_departement"]
].dropna(subset=["code_insee_commune"]).copy()

# Nettoyage du code INSEE (format texte propre à 5 caractères)
df_communes["code_insee"] = (
    df_communes["code_insee_commune"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

df_communes = (
    df_communes[["code_insee", "consolidated_commune", "consolidated_code_postal", "code_departement"]]
    .drop_duplicates(subset=["code_insee"])
    .rename(columns={"consolidated_commune": "nom_commune", "consolidated_code_postal": "code_postal"})
)
df_communes.to_sql("COMMUNES", conn, if_exists="append", index=False)


# INJECTION TABLE 2 : OPERATEURS (Parente)

df_operateurs = (
    df_clean[["nom_amenageur", "nom_operateur"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
df_operateurs.to_sql("OPERATEURS", conn, if_exists="append", index=False)

# Rapatrier les identifiants générés par SQLite pour les attribuer aux stations
df_operateurs_db = pd.read_sql("SELECT * FROM OPERATEURS", conn)
df_clean_merged = df_clean.merge(df_operateurs_db, on=["nom_amenageur", "nom_operateur"], how="left")

# INJECTION TABLE 3 : STATIONS (Enfant de COMMUNES et OPERATEURS)

df_stations = df_clean_merged[
    [
        "id_station_itinerance",
        "code_insee_commune",
        "id_operateur",
        "nom_station",
        "adresse_station",
        "consolidated_latitude",
        "consolidated_longitude",
        "nbre_pdc",
    ]
].dropna(subset=["id_station_itinerance", "code_insee_commune", "id_operateur"]).copy()

df_stations["code_insee"] = (
    df_stations["code_insee_commune"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

df_stations = df_stations.drop_duplicates(subset=["id_station_itinerance"]).rename(
    columns={
        "id_station_itinerance": "id_station",
        "consolidated_latitude": "latitude",
        "consolidated_longitude": "longitude",
    }
)

# Sécuriser l'intégrité des données : filtre sur les clés existantes dans COMMUNES
valid_communes = set(df_communes["code_insee"])
df_stations = df_stations[df_stations["code_insee"].isin(valid_communes)]
df_stations["id_operateur"] = df_stations["id_operateur"].astype(int)

df_stations[
    ["id_station", "code_insee", "id_operateur", "nom_station", "adresse_station", "latitude", "longitude", "nbre_pdc"]
].to_sql("STATIONS", conn, if_exists="append", index=False)

# INJECTION TABLE 4 : POINTS_DE_CHARGE (Enfant de STATIONS)

df_pdc = df_clean_merged[
    [
        "id_pdc_itinerance",
        "id_station_itinerance",
        "puissance_nominale",
        "tranche_puissance",
        "prise_type_2",
        "prise_type_combo_ccs",
    ]
].dropna(subset=["id_pdc_itinerance", "id_station_itinerance"]).copy()

df_pdc = df_pdc.drop_duplicates(subset=["id_pdc_itinerance"]).rename(
    columns={
        "id_pdc_itinerance": "id_pdc",
        "id_station_itinerance": "id_station",
    }
)

# Sécurité d'intégrité : filtre sur les stations réellement insérées
valid_stations = set(df_stations["id_station"])
df_pdc = df_pdc[df_pdc["id_station"].isin(valid_stations)]

df_pdc.to_sql("POINTS_DE_CHARGE", conn, if_exists="append", index=False)

# Validation définitive de la transaction
conn.commit()

# BILAN ET VÉRIFICATION

print("\n--- BASE DE DONNÉES INJECTÉE AVEC SUCCÈS ---")
for table in ["COMMUNES", "OPERATEURS", "STATIONS", "POINTS_DE_CHARGE"]:
    count = pd.read_sql(f"SELECT COUNT(*) as total FROM {table}", conn)["total"].iloc[0]
    print(f"Table {table:<18} : {count:,} lignes")

conn.close()

Schéma de la base de données créé avec succès.

--- BASE DE DONNÉES INJECTÉE AVEC SUCCÈS ---
Table COMMUNES           : 11,703 lignes
Table OPERATEURS         : 4,412 lignes
Table STATIONS           : 48,581 lignes
Table POINTS_DE_CHARGE   : 166,185 lignes


In [4]:
# Bilan des colonnes avec valeurs nulles
null_counts = df_clean.isna().sum()
cols_nulles = null_counts[null_counts > 0]

df_diag_nulls = pd.DataFrame({
    "Manquants (Absolu)": cols_nulles,
    "Pourcentage (%)": (df_clean[cols_nulles.index].isna().mean() * 100).round(2)
}).sort_values(by="Manquants (Absolu)", ascending=False)

df_diag_nulls

,Manquants (Absolu),Pourcentage (%)
nom_amenageur,1389,0.84
nom_operateur,413,0.25
